<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_review_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Review segmentation

## Set up

In [35]:
# @title Dependencies

# Installation
!pip install strsimpy -q
!pip install pandarallel -q

# First-party installations
import os
from google.colab import drive
from IPython.display import display
import re
import hashlib
import json

# Third-party installations
import pandas as pd
import numpy as np
from tqdm import tqdm
from strsimpy.normalized_levenshtein import NormalizedLevenshtein
from pandarallel import pandarallel

In [36]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR
DATASET_DIR = os.path.join(WORKING_DIR, "llm_reviews/generated")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews/segmented")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "results.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "results.jsonl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/generated.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/segmented.


In [37]:
# @title Setup definitions

INPUT_DATA = "results.parquet"

TARGET_SECTION = "weaknesses"

# Define target columns
REVIEW_COL_ORIGINAL = 'llm_review'
REVIEW_COL_EXTRACTED_WEAKNESS = 'extracted_weaknesses'
REVIEW_COL_EXTRACTED_RATING = 'extracted_scores'
REVIEW_COL_SEGMENTS = 'segmented_comment'

# Define id columns
CONFIG_UNI_ID = "config_group"
UNI_ID = "segment_hash"

# Define config columns for grouping hash
config_columns = [
    "model",
    "temperature",
    "reasoning_effort",
    "chain_of_thought",
    "note_taking"
]

# Define the desired order of columns
COL_ORDER = [
    UNI_ID,
    CONFIG_UNI_ID,
    'paper_id',
    'run_signature',
    'model',
    'temperature',
    'reasoning_effort',
    'chain_of_thought',
    'note_taking',
    'llm_notes',
    'llm_review',
    REVIEW_COL_EXTRACTED_WEAKNESS,
    REVIEW_COL_EXTRACTED_RATING,
    REVIEW_COL_SEGMENTS,
    'correctness_rating',
    'technical_novelty_significance_rating',
    'empirical_novelty_significance_rating',
    'generation_successful'
]

# Define keys for numerical ratings
RATING_KEYS = [
    "correctness_rating",
    "technical_novelty_significance_rating",
    "empirical_novelty_significance_rating"
]

# Define specific error strings for review generation/extraction/parsing failures
REVIEW_GENERATION_FAILURE = "ERROR: REVIEW GENERATION NOT SUCCESSFUL"
PARALLEL_PROCESSING_FAILURE = "ERROR: UNKNOWN STATE REACHED IN PARALLEL PROCESSING"
EXTRACTION_FAILURE = "ERROR: SEGMENT EXTRACTION FAILED"

In [38]:
# @title Data import

# Read dataset
df = pd.read_parquet(os.path.join(DATASET_DIR, INPUT_DATA))

# Check and display
display(df.shape)
display(df.head(3))

(208, 9)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,44938bfcb50e,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""The paper studies applyi..."
1,WoByU5W5te0,gpt-5-mini-2025-08-07,NaN,low,,Faithful,44938bfcb50e,Title\n- GeCoNeRF: Few-Shot Neural Radiance Fi...,"{""summary_of_paper"": ""This paper proposes GeCo..."
2,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,44938bfcb50e,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies the u..."


In [39]:
# @title Configuration group identifier

# Create an informative identifier for each configuration group
df[CONFIG_UNI_ID] = df.apply(lambda row: ' | '.join([f'{col}: {row[col]}' for col in config_columns]), axis=1)

# Check and display grouping identifier(s)
display(df.shape)
display(df.head(3))

(208, 10)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review,config_group
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,44938bfcb50e,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""The paper studies applyi...",model: gpt-5-mini-2025-08-07 | temperature: na...
1,WoByU5W5te0,gpt-5-mini-2025-08-07,NaN,low,,Faithful,44938bfcb50e,Title\n- GeCoNeRF: Few-Shot Neural Radiance Fi...,"{""summary_of_paper"": ""This paper proposes GeCo...",model: gpt-5-mini-2025-08-07 | temperature: na...
2,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,44938bfcb50e,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies the u...",model: gpt-5-mini-2025-08-07 | temperature: na...


## Define

In [40]:
# @title Extraction logic

def extract_weaknesses(json_str):
    """
    Parses the json_str and extracts the TARGET_SECTION section.
    Returns a list of strings, or a list containing an error string if parsing fails or the section is missing.
    """
    # Handle specific error strings first
    if json_str in [REVIEW_GENERATION_FAILURE, PARALLEL_PROCESSING_FAILURE]:
        return [json_str]

    try:
        # Load columnn data into JSON
        review_data = json.loads(json_str)

        # Isolate the TARGET_SECTION
        target_section = review_data.get(TARGET_SECTION)

        # If target_section is None or empty, return the specific error string
        if not target_section:
            return [EXTRACTION_FAILURE]

        # Return the whole section
        return target_section

    except Exception as e:
        # Print a hardcoded error message for debugging
        print(f"Warning: Failed to isolate weaknesses. Error: {e}. Review string: {json_str[:100]}...")
        return [EXTRACTION_FAILURE]

def extract_ratings(json_str, rating_keys):
    """
    Parses the json_str and extracts numerical ratings for specified keys.
    Returns a dictionary of ratings, or None for missing keys.
    If the input is an error string, all ratings will be that error string.
    """
    ratings = {}
    # Handle specific error strings first
    if json_str in [REVIEW_GENERATION_FAILURE, PARALLEL_PROCESSING_FAILURE]:
        for key in rating_keys:
            ratings[key] = None # Assign None for numerical rating errors
        return ratings

    try:
        review_data = json.loads(json_str)
        for key in rating_keys:
            ratings[key] = review_data.get(key)
    except Exception as e:
        print(f"Warning: Failed to extract ratings. Error: {e}. Review string: {json_str[:100]}...")
        for key in rating_keys:
            ratings[key] = None # Assign None for numerical rating errors
    return ratings

## Run

In [41]:
# @title Extraction

# Apply the extraction function for weaknesses
df[REVIEW_COL_EXTRACTED_WEAKNESS] = df[REVIEW_COL_ORIGINAL].apply(extract_weaknesses)

# Apply the extraction function for ratings
extracted_ratings_series = df[REVIEW_COL_ORIGINAL].apply(lambda x: extract_ratings(x, RATING_KEYS))

# Add the full dictionary of ratings as a new column
df[REVIEW_COL_EXTRACTED_RATING] = extracted_ratings_series

# Expand the dictionary of ratings into separate columns
for key in RATING_KEYS:
    df[key] = extracted_ratings_series.apply(lambda x: x.get(key))

# Check and display extracted section(s) and new rating columns
display(df.shape)
display(df.head())

  "summary_of_paper": "The paper introduces the concept of Minimum Variance Unbiased N:M Sparsity ...
  "summary_of_paper": "This paper presents GeCoNeRF, a method to enhance Neural Radiance Fields (N...
  "summary_of_paper": "The paper proposes advancing N:M structured sparsity by introducing an unbi...
  "summary_of_paper": "The paper introduces GeCoNeRF, a new framework to regularize Neural Radianc...
  "summary_of_paper": "The paper presents a novel framework grounded in transformer-inspired archi...
  "summary_of_paper": "The paper proposes a novel methodology to implement sharp N:M fine-grained ...
  "summary_of_paper": "The paper introduces GeCoNeRF, a regularization method for training Neural ...
  "summary_of_paper": "The paper investigates the application of fine-scale N:M structured sparsit...
  "summary_of_paper": "This paper investigates the application of N:M fine-grained structured spar...
  "summary_of_paper": "This paper investigates the application of N:M fine-grained

(208, 15)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review,config_group,extracted_weaknesses,extracted_scores,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,44938bfcb50e,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""The paper studies applyi...",model: gpt-5-mini-2025-08-07 | temperature: na...,[No end-to-end wall-clock training speedups ar...,"{'correctness_rating': 5, 'technical_novelty_s...",5,4,4
1,WoByU5W5te0,gpt-5-mini-2025-08-07,NaN,low,,Faithful,44938bfcb50e,Title\n- GeCoNeRF: Few-Shot Neural Radiance Fi...,"{""summary_of_paper"": ""This paper proposes GeCo...",model: gpt-5-mini-2025-08-07 | temperature: na...,[Some important hyperparameters and loss weigh...,"{'correctness_rating': 4, 'technical_novelty_s...",4,3,4
2,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,44938bfcb50e,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies the u...",model: gpt-5-mini-2025-08-07 | temperature: na...,[No native hardware support yet; the paper can...,"{'correctness_rating': 4, 'technical_novelty_s...",4,4,3
3,WoByU5W5te0,gpt-5-mini-2025-08-07,NaN,low,,Turned off,44938bfcb50e,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper proposes GeCo...",model: gpt-5-mini-2025-08-07 | temperature: na...,[Method depends on the quality of the currentl...,"{'correctness_rating': 4, 'technical_novelty_s...",4,3,3
4,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,Explain your thought process step by step.,Faithful,44938bfcb50e,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""This paper extends fine-...",model: gpt-5-mini-2025-08-07 | temperature: na...,[Limited runtime/throughput validation: due to...,"{'correctness_rating': 5, 'technical_novelty_s...",5,4,4


In [42]:
# @title Success metric

# Add metric column for later analysis
df['generation_successful'] = ~df[REVIEW_COL_EXTRACTED_WEAKNESS].isin([REVIEW_GENERATION_FAILURE, PARALLEL_PROCESSING_FAILURE, EXTRACTION_FAILURE])

## Transformation

In [43]:
# @title Long-format

# Create a copy of the extracted section column
df[REVIEW_COL_SEGMENTS] = df[REVIEW_COL_EXTRACTED_WEAKNESS]

# Convert the copied column into long-format
df = df.explode(REVIEW_COL_SEGMENTS)

# Display and check segmented comment(s)
display(df.shape)
display(df.head(3))

(619, 17)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review,config_group,extracted_weaknesses,extracted_scores,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating,generation_successful,segmented_comment
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,44938bfcb50e,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""The paper studies applyi...",model: gpt-5-mini-2025-08-07 | temperature: na...,[No end-to-end wall-clock training speedups ar...,"{'correctness_rating': 5, 'technical_novelty_s...",5,4,4,True,No end-to-end wall-clock training speedups are...
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,44938bfcb50e,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""The paper studies applyi...",model: gpt-5-mini-2025-08-07 | temperature: na...,[No end-to-end wall-clock training speedups ar...,"{'correctness_rating': 5, 'technical_novelty_s...",5,4,4,True,Exact MVUE 2:4 implementation is computational...
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,44938bfcb50e,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""The paper studies applyi...",model: gpt-5-mini-2025-08-07 | temperature: na...,[No end-to-end wall-clock training speedups ar...,"{'correctness_rating': 5, 'technical_novelty_s...",5,4,4,True,The work assumes unbiasedness is strictly nece...


In [44]:
# @title Unique identifiers

# Get all columns as basis for unique identifier (hash)
col_for_hash = df.columns.tolist()

# Create a unique identifier (hash)
df[UNI_ID] = df[col_for_hash].astype(str).agg(''.join, axis=1).apply(lambda x: hashlib.sha1(x.encode('utf-8')).hexdigest()[:12])

# Display and check unique identifier(s)
display(df.shape)
display(df.head(3))

(619, 18)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review,config_group,extracted_weaknesses,extracted_scores,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating,generation_successful,segmented_comment,segment_hash
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,44938bfcb50e,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""The paper studies applyi...",model: gpt-5-mini-2025-08-07 | temperature: na...,[No end-to-end wall-clock training speedups ar...,"{'correctness_rating': 5, 'technical_novelty_s...",5,4,4,True,No end-to-end wall-clock training speedups are...,ce78cd3ae777
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,44938bfcb50e,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""The paper studies applyi...",model: gpt-5-mini-2025-08-07 | temperature: na...,[No end-to-end wall-clock training speedups ar...,"{'correctness_rating': 5, 'technical_novelty_s...",5,4,4,True,Exact MVUE 2:4 implementation is computational...,1f58b1da7892
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,44938bfcb50e,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""The paper studies applyi...",model: gpt-5-mini-2025-08-07 | temperature: na...,[No end-to-end wall-clock training speedups ar...,"{'correctness_rating': 5, 'technical_novelty_s...",5,4,4,True,The work assumes unbiasedness is strictly nece...,423e2e9fcc52


In [45]:
# @title Reorder columns

# Reindex the dataset to apply the new column order
df = df.reindex(columns=COL_ORDER)

# Display and check the reordered dataset
display(df.shape)
display(df.head(3))

(619, 18)

,segment_hash,config_group,paper_id,run_signature,model,temperature,reasoning_effort,chain_of_thought,note_taking,llm_notes,llm_review,extracted_weaknesses,extracted_scores,segmented_comment,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating,generation_successful
0,ce78cd3ae777,model: gpt-5-mini-2025-08-07 | temperature: na...,vuD2xEtxZcj,44938bfcb50e,gpt-5-mini-2025-08-07,NaN,low,,Faithful,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""The paper studies applyi...",[No end-to-end wall-clock training speedups ar...,"{'correctness_rating': 5, 'technical_novelty_s...",No end-to-end wall-clock training speedups are...,5,4,4,True
0,1f58b1da7892,model: gpt-5-mini-2025-08-07 | temperature: na...,vuD2xEtxZcj,44938bfcb50e,gpt-5-mini-2025-08-07,NaN,low,,Faithful,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""The paper studies applyi...",[No end-to-end wall-clock training speedups ar...,"{'correctness_rating': 5, 'technical_novelty_s...",Exact MVUE 2:4 implementation is computational...,5,4,4,True
0,423e2e9fcc52,model: gpt-5-mini-2025-08-07 | temperature: na...,vuD2xEtxZcj,44938bfcb50e,gpt-5-mini-2025-08-07,NaN,low,,Faithful,"Paper: ""Minimum Variance Unbiased N:M Sparsity...","{""summary_of_paper"": ""The paper studies applyi...",[No end-to-end wall-clock training speedups ar...,"{'correctness_rating': 5, 'technical_novelty_s...",The work assumes unbiasedness is strictly nece...,5,4,4,True


In [46]:
# @title Results

# Convert all columns to strings (ensures the dataset can be saved to .parquet)
df = df.map(lambda x: str(x) if x is not None else None)

# Save the long-format dataset to JSONL
df.to_json(RESULTS_JSON_PATH, orient='records', lines=True)
print(f"Dataset saved to JSONL at: {RESULTS_JSON_PATH}")

# Save the long-format Dataset to Parquet
df.to_parquet(RESULTS_PATH, index=False)
print(f"Dataset saved to Parquet at: {RESULTS_PATH}")

Dataset saved to JSONL at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/segmented/results.jsonl
Dataset saved to Parquet at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/segmented/results.parquet
